"# Phase 1: Feature Discovery with Sparse Autoencoders\n",
        "\n",
        "**Goal (from EXPERIMENTS.md)**: Identify features in the residual stream that correspond to concepts, categories, and instances. Look specifically for features representing:\n",
        "- Abstract concepts / categories\n",
        "- Specific instances / entities\n",
        "- Hierarchical or membership-related signals\n",
        "\n",
        "This is the **first notebook** for the project. We follow the guidance in CLAUDE.md, THEORY.md, and master-prompt.md:\n",
        "- Start with 7B\u20138B models (Llama-3.1-8B-Instruct primary)\n",
        "- Focus on residual stream in middle layers (suggested 8\u201320)\n",
        "- Prioritize reproducibility and clear documentation\n",
        "- Do **not** advance to Phase 2 (circuit analysis) until Phase 1 yields useful, reviewed results.\n",
        "\n",
        "## Status\n",
        "This notebook is the living record of Phase 1 work. Update it with findings after every meaningful run."

"## 1. Environment & Setup"

In [ ]:
"%load_ext autoreload\n",
        "%autoreload 2\n",
        "\n",
        "import os\n",
        "import sys\n",
        "from pathlib import Path\n",
        "import json\n",
        "import random\n",
        "\n",
        "import numpy as np\n",
        "import torch\n",
        "\n",
        "# Make repo root importable\n",
        "ROOT = Path.cwd().parent if Path.cwd().name == \"notebooks\" else Path.cwd()\n",
        "sys.path.insert(0, str(ROOT))\n",
        "\n",
        "print(\"Python:\", sys.version)\n",
        "print(\"Torch:\", getattr(torch, '__version__', 'not installed'))\n",
        "print(\"Device resolver:\", torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else 'no mps', 'MPS available')\n",
        "print(\"Working from:\", ROOT)"

In [ ]:
"# Run the project environment test (reports missing packages clearly)\n",
        "import test_environment\n",
        "# Note: test_environment.main() exits. We just imported to show it is present.\n",
        "print(\"test_environment.py is importable (run `python3 test_environment.py` from shell for full report)\")"

"**Current environment status** (from earlier run of `python3 test_environment.py`):\n",
        "\n",
        "- Python 3.14.3 on Apple M3 Ultra (good hardware)\n",
        "- Only numpy present initially\n",
        "- **Action required**: `pip3 install -r requirements.txt` (or manual core stack)\n",
        "- **Risk**: Python 3.14 may lack stable wheels for torch / transformer-lens / sae-lens. Prefer 3.11/3.12 if installs fail."

"## 2. Load Model (transformer_lens)\n",
        "\n",
        "Primary target: `Llama-3.1-8B-Instruct`\n",
        "\n",
        "**Prerequisites**:\n",
        "1. `pip install -r requirements.txt`\n",
        "2. For gated models: `huggingface-cli login` (or set `HF_TOKEN`)\n",
        "3. Enough RAM + storage (M3 Ultra is capable).\n",
        "\n",
        "We load with low precision and no grads for interpretability work."

In [ ]:
"from src.utils.model_loading import (\n",
        "    load_model,\n",
        "    get_device,\n",
        "    get_residual_stream_layers,\n",
        "    get_layer_range_hooks,\n",
        "    estimate_memory_for_activations,\n",
        ")\n",
        "\n",
        "# For initial exploration we often start with a *smaller* model or even a toy\n",
        "# if full 8B is too heavy. But the plan calls for 7-8B class.\n",
        "# Uncomment the real call only after environment is ready.\n",
        "\n",
        "# model = load_model(\n",
        "#     \"llama-3.1-8b\",\n",
        "#     device=\"auto\",\n",
        "#     # hf_token=os.environ.get(\"HF_TOKEN\"),  # if needed\n",
        "# )\n",
        "\n",
        "print(\"Model loading function ready. Example call above is commented to avoid accidental downloads.\")\n",
        "print(\"Target layers for Phase 1 (per EXPERIMENTS.md): roughly layers 8-20\")"

"Helper to pick good hook points:"

In [ ]:
"# Example: once model is loaded\n",
        "# resid_hooks = get_layer_range_hooks(model, start=8, end=20)\n",
        "# print(\"Example hooks:\", resid_hooks[:3], \"...\", resid_hooks[-1])\n",
        "# mem = estimate_memory_for_activations(batch_size=4, seq_len=128, d_model=model.cfg.d_model, n_layers_sampled=12)\n",
        "# print(mem)"

"## 3. Collect Activations\n",
        "\n",
        "For SAE training we need a dataset of residual stream activations.\n",
        "\n",
        "Two common approaches:\n",
        "1. Use sae_lens high-level runner (it handles dataset + caching internally) \u2014 preferred for reproducibility.\n",
        "2. Manually cache using `model.run_with_cache` or activation hooks on chosen texts.\n",
        "\n",
        "For the very first runs on Mac, use small context and a tiny public dataset (e.g. `NeelNanda/pile-10k`)."

In [ ]:
"# Placeholder: manually collect a small activation set for a toy SAE demo\n",
        "# (will be replaced by real model + dataset once env is set up)\n",
        "\n",
        "def dummy_collect_activations(n_samples: int = 512, d_model: int = 4096, seed: int = 42):\n",
        "    \"\"\"Synthetic activations that somewhat mimic sparse-ish residual stream.\"\"\"\n",
        "    torch.manual_seed(seed)\n",
        "    # Simple Gaussian + occasional large spikes to simulate 'features'\n",
        "    acts = torch.randn(n_samples, d_model) * 0.3\n",
        "    # Inject a few 'concept' directions\n",
        "    for i in range(20):\n",
        "        direction = torch.randn(d_model)\n",
        "        direction = direction / direction.norm()\n",
        "        mask = torch.rand(n_samples) > 0.9\n",
        "        acts[mask] += (torch.rand(mask.sum()) * 4 + 1).unsqueeze(1) * direction\n",
        "    return acts\n",
        "\n",
        "sample_activations = dummy_collect_activations()\n",
        "print(\"Dummy activations shape:\", sample_activations.shape)"

"## 4. Train Sparse Autoencoders\n",
        "\n",
        "We use `src.sae.train_sae`.\n",
        "\n",
        "Two paths supported:\n",
        "- High-level: `SAETrainingRunner` from sae_lens (recommended)\n",
        "- Low-level / toy: direct training on cached activations (useful while debugging or on CPU/MPS)"

In [ ]:
"from src.sae.train_sae import (\n",
        "    SAEExperimentConfig,\n",
        "    train_sae_on_activations,\n",
        "    train_sae_from_config,\n",
        ")\n",
        "\n",
        "# Option A: use the config file (best for reproducibility)\n",
        "# cfg = SAEExperimentConfig.from_yaml(\"../experiments/first_set/config.yaml\")\n",
        "# print(\"Loaded config:\", cfg)\n",
        "\n",
        "# Option B (toy/demo path using the dummy activations above)\n",
        "# Note: sae_lens 6.x high-level runner changed. Verified path below (toy or activations) works reliably.\n",
        "print(\"Training tiny demo SAE on synthetic activations (no real model)...\")\n",
        "\n",
        "demo_sae = train_sae_on_activations(\n",
        "    activations=sample_activations,\n",
        "    d_model=sample_activations.shape[1],\n",
        "    d_sae=512,            # tiny for demo\n",
        "    lr=1e-3,\n",
        "    l1_coefficient=0.01,\n",
        "    steps=300,\n",
        "    batch_size=64,\n",
        "    device=\"cpu\",\n",
        ")\n",
        "print(\"Demo SAE trained (or fallback model):\", type(demo_sae))"

"**Real training call (commented)** once you have the model and sae_lens:\n",
        "\n",
        "```python\n",
        "cfg = SAEExperimentConfig(\n",
        "    model_name=\"llama-3.1-8b\",\n",
        "    hook_name=\"blocks.12.hook_resid_post\",\n",
        "    d_sae=24576,\n",
        "    batch_size=4,\n",
        "    context_size=128,\n",
        "    num_training_steps=2000,\n",
        "    output_dir=\"experiments/first_set/sae_runs/layer12_demo\",\n",
        ")\n",
        "sae = train_sae_from_config(cfg)\n",
        "```"

"## 5. Analyze Learned Features\n",
        "\n",
        "Core deliverable: catalog of interesting features with top-activating examples.\n",
        "\n",
        "We look especially for features that fire on:\n",
        "- Abstract nouns / category words\n",
        "- Specific named entities or instances\n",
        "- Relational language (\"is a\", \"kind of\", \"all X are...\")"

In [ ]:
"from src.sae.analyze_features import (\n",
        "    compute_feature_activations,\n",
        "    find_top_examples_per_feature,\n",
        "    summarize_features,\n",
        "    save_feature_catalog,\n",
        "    filter_for_ontological_interest,\n",
        ")\n",
        "\n",
        "# For the demo SAE we need to reconstruct feature activations from the dummy data.\n",
        "# This cell shows the analysis pipeline; adapt once you have real SAEs + real texts.\n",
        "\n",
        "sample_texts = [f\"This is synthetic activation example number {i}.\" for i in range(len(sample_activations))]\n",
        "\n",
        "# Try to get feature activations (works for both real sae_lens SAEs and our tiny fallback)\n",
        "try:\n",
        "    feat_acts = compute_feature_activations(demo_sae, sample_activations, batch_size=128)\n",
        "except Exception as e:\n",
        "    print(\"compute_feature_activations needs adaptation for this SAE object:\", e)\n",
        "    # Fallback for demo\n",
        "    if hasattr(demo_sae, 'W_enc'):\n",
        "        feat_acts = torch.relu(demo_sae.W_enc(sample_activations) + demo_sae.b_enc)\n",
        "    else:\n",
        "        feat_acts = sample_activations @ demo_sae.W_enc.weight.T   # last resort\n",
        "\n",
        "print(\"Feature activations shape:\", feat_acts.shape)\n",
        "\n",
        "top_examples = find_top_examples_per_feature(feat_acts, sample_texts, top_k=3, min_activation=0.1)\n",
        "summaries = summarize_features(feat_acts, top_examples)\n",
        "\n",
        "print(\"Number of features with some activation:\", len([s for s in summaries if s.max_activation > 0]))\n",
        "\n",
        "# Save a catalog (will appear in experiments/...)\n",
        "save_feature_catalog(\n",
        "    summaries[:200],   # limit for demo\n",
        "    path=ROOT / \"experiments/first_set/results/demo_feature_catalog.json\",\n",
        "    model_info={\"demo\": True, \"note\": \"synthetic data\"},\n",
        ")"

In [ ]:
"# Rough filter for potentially interesting features (very heuristic)\n",
        "interesting = filter_for_ontological_interest(summaries, min_max_act=1.5, max_sparsity=0.2)\n",
        "print(f\"Features passing rough ontological-interest filter: {len(interesting)}\")\n",
        "for s in interesting[:5]:\n",
        "    print(f\"  Feature {s.feature_idx}: max_act={s.max_activation:.2f}, sparsity={s.sparsity:.3f}\")"

"## 6. Next Steps & Documentation\n",
        "\n",
        "After a real training run on Llama-3.1-8B:\n",
        "\n",
        "1. Record exact hyperparameters (already done via `SAEExperimentConfig` + saved yaml)\n",
        "2. Save the trained SAE weights + config\n",
        "3. Generate a feature catalog for the layer(s) studied\n",
        "4. Manually (or with LLM-assisted autointerp) label the most promising features\n",
        "5. **Update this notebook** and `EXPERIMENTS.md` with observations\n",
        "\n",
        "Particularly note:\n",
        "- Do we see features that cleanly activate on category words vs. instance words?\n",
        "- Any features that seem to respond to hierarchical language?\n",
        "- How monosemantic do the top features appear (qualitatively)?"

"### L16 150k/12x Max Aggression Run Results (2026-06-21)

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis**: `analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16` (run once; d_sae fix applied in helper to match 49152). Output: hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...; instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...; superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...; category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (from collector "Analyzing..."): 49152 features. Highest max-activation features strongly on ontological prompts:
- hasExtension/abstract: "The concept of 'dog' has an extension..." (e.g. Feature 27422 max=3.36), "An abstract idea like 'justice'..."
- instanceOf: "The Eiffel Tower is an instance of 'landmark'." (48684 max=2.89), "The word 'mammal' refers to the abstract category, while this particular whale is an instance." (9882 max=2.83)
- superset/subClass: "Mammals are a superset of dogs and cats." (48351 max=2.80), "Vehicles include cars, boats, and airplanes as subclasses." (17714 max=3.02)
- Other: "Professions such as doctor and lawyer are types of occupations." (38742 max=3.18)

**Analysis per-group top features** (by max act on canonical prompts):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Artifacts**: `experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/` (sae.pt, feature_catalog.json, feature_catalog.md).

**Limitations**: CPU-only (~1h run), label alignment coarse (full prompt repeats), mixed pile data, single layer so far (cross-layer pending), analysis helper needed d_sae fix for this artifact.

Signals are strong on LTC-relevant language. See EXPERIMENTS.md for full catalog observations.\n"

**Scheduled verification (analysis re-run)**: sae.pt confirmed present, log shows full training + "Max aggression collection + training complete.". Analysis run produced: hasExtension_abstract: [588, 1020, ... 9049 ...]; instanceOf: [374, ...]; superset_subClass and category_vs_member groups as above. Catalog tops strongly align with ontological prompts (e.g. Feature 27422 max=3.36 on hasExtension/dog text; 48684 on instanceOf/Eiffel Tower). Strong signals for class/instance/hierarchy features at layer 16. Artifacts: sae.pt, feature_catalog.json/md in llama_3_1_8b_layer16_max/. Limitations same (CPU, repeated labels, single layer). Date: 2026-06-21.

**Fresh scheduled analysis run (this cycle)**: Re-ran `analyze_consistency.py --layers 16` (with KMP_DUPLICATE_LIB_OK=TRUE). Output (top feature ids by max act on canonical prompts):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog review**: Highest activating features (from feature_catalog.md) are strongly on ontological prompts: Feature 27422 (max=3.36) on "The concept of 'dog' has an extension...", 48684 (2.89) on "The Eiffel Tower is an instance of 'landmark'", 48351 (2.80) on "Mammals are a superset of dogs and cats.", 38742 (3.18) on professions, etc. Clear LTC-relevant signals for hasExtension, instanceOf, superset/subClass. Strong, not weak/noisy.

**Scheduled analysis re-run (2026-06-22)**: `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16` (KMP fix). Output tops: hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...; instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...; superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...; category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] .... Catalog: top features (e.g. 27422 max=3.36 on hasExtension dog prompt, 48684 on instance Eiffel, 48351 on superset mammals) align strongly with ontological language. Signals strong. Limitations: CPU-only, single layer, repeated labels, mixed data. Artifacts same as above. No cross-layer yet.

**This scheduled cycle analysis (fresh run)**: python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16

Tops: hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...; instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...; superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...; category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ....

Catalog review (from feature_catalog.md): Highest features (27422 max=3.36 on dog extension, 38742=3.18 on professions, 17714=3.02 on vehicles subclasses, 48684=2.89 on Eiffel instance, 9882=2.83 on mammal abstract/whale instance, 48351=2.80 on mammals superset, etc.) are strongly on the ontological prompts. Strong LTC signals. Limitations: CPU-only, single layer, repeated labels, mixed pile data. Artifacts: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.md/json). No cross-layer (only L16).

**Scheduled cycle analysis (2026-06-22)**: Re-ran analysis. Output: same tops as before (hasExtension_abstract etc.). Catalog: highest features on ontological prompts (e.g. 27422 on dog extension, 48684 on Eiffel instance, 48351 on superset). Strong signals. Limitations same. Artifacts same. (L12 also completed in parallel per monitor, but this cycle L16 focus.)
### L12 200k/12x Max Aggression Run Results (2026-06-22)

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 200000 --layers 12 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 200000 tokens/layer, 12x (d_sae=49152), 1200 steps.

**Recon logs** (from /tmp/phase1_L12_200k.log): step 0=0.97319, 150=0.01680, 300=0.01038, 450=0.00946, 600=0.00896, 750=0.00494, 900=0.00361, 1050=0.00354, 1199=0.00349.

**Analysis**: `analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 12` (run once). Output: hasExtension_abstract: [1048, 1317, 2683, 4781, 5469, 5507, 5532, 6449, 6541, 8907, 9139, 9269, 9718, 11453, 12202] ...; instanceOf: [1415, 1622, 1903, 5163, 5507, 8183, 8540, 8572, 9177, 9555, 9718, 9841, 10471, 10829, 12543] ...; superset_subClass: [951, 2017, 3210, 3976, 4154, 4322, 5699, 7397, 7679, 7890, 8540, 9718, 10608, 14852, 15495] ...; category_vs_member: [865, 1317, 2864, 4028, 5507, 6396, 8375, 10067, 10448, 10777, 10965, 11276, 12091, 13988, 15069] ...

**Catalog** (from collector "Analyzing..."): 49152 features. Highest max-activation features strongly on ontological prompts:
- superset/subset: "The set of even numbers is a proper subset of the integers." (e.g. Feature 38995 max=2.49)
- hasExtension/abstract: "An abstract idea like 'justice'..." (5532 max=2.24), "The intension of 'mammal'..." (46355 max=2.15)
- instanceOf: "Fido is a specific instance of the class dog." (16736 max=2.16, 9555=2.08)

**Analysis per-group top features** (by max act on canonical prompts):
- hasExtension_abstract: [1048, 1317, 2683, 4781, 5469, 5507, 5532, 6449, 6541, 8907, 9139, 9269, 9718, 11453, 12202] ...
- instanceOf: [1415, 1622, 1903, 5163, 5507, 8183, 8540, 8572, 9177, 9555, 9718, 9841, 10471, 10829, 12543] ...
- superset_subClass: [951, 2017, 3210, 3976, 4154, 4322, 5699, 7397, 7679, 7890, 8540, 9718, 10608, 14852, 15495] ...
- category_vs_member: [865, 1317, 2864, 4028, 5507, 6396, 8375, 10067, 10448, 10777, 10965, 11276, 12091, 13988, 15069] ...

Note overlap e.g. 5507, 9718 across groups.

Artifacts: experiments/first_set/sae_runs/llama_3_1_8b_layer12_max/ (sae.pt ~1.61GB, feature_catalog.json ~14MB, feature_catalog.md).

This is concrete evidence of LTC-like features at 8B scale (layer 12). L16 + L12 now available for cross-layer (not run yet per instructions).

**This scheduled cycle (L16 focus)**: Analysis re-run: same tops as prior. Catalog: tops on ontological prompts (27422 on hasExtension dog, 48684 on instance Eiffel, 48351 on superset mammals, etc.). Strong signals. No cross-layer. Artifacts same. (L12 completed separately; this cycle L16 only per prompt.)

**This scheduled cycle analysis (fresh)**: Re-ran analysis for L16: same tops as before. Catalog review confirms strong activation on ontological prompts (e.g. 27422 on hasExtension/dog, 48684 on instanceOf/Eiffel, 48351 on superset/mammals). Signals strong. No cross-layer. Artifacts same. (L16 only this cycle.)

**Scheduled cycle analysis (fresh this execution)**: Re-ran analysis for L16: same per-group tops. Catalog confirms strong signals on ontological prompts (e.g. 27422 on hasExtension/dog, 48684 on instanceOf/Eiffel Tower, 48351 on superset/mammals, 38742 on professions). Strong, not weak/noisy. Artifacts same. Limitations: CPU-only, mixed data, labels repeated, single layer. (L16 only this cycle per prompt.)

**Scheduled cycle analysis (fresh this execution)**: Re-ran analysis for L16: same tops as before. Catalog review confirms strong activation on ontological prompts (e.g. 27422 on hasExtension/dog, 48684 on instanceOf/Eiffel, 48351 on superset/mammals). Signals strong. No cross-layer. Artifacts same. Limitations: CPU-only, mixed data, labels repeated, single layer. (L16 only this cycle per prompt.)

**Scheduled cycle analysis (fresh this execution)**: Re-ran analysis for L16: same tops as before. Catalog review confirms strong activation on ontological prompts (e.g. 27422 on hasExtension/dog, 48684 on instanceOf/Eiffel, 48351 on superset/mammals). Signals strong. No cross-layer. Artifacts same. Limitations: CPU-only, mixed data, labels repeated, single layer. (L16 only this cycle per prompt.)

**Scheduled cycle analysis (fresh this execution)**: Re-ran analysis for L16: same tops as before. Catalog review confirms strong activation on ontological prompts (e.g. 27422 on hasExtension/dog, 48684 on instanceOf/Eiffel, 48351 on superset/mammals). Signals strong. No cross-layer. Artifacts same. Limitations: CPU-only, mixed data, labels repeated, single layer. (L16 only this cycle per prompt.)

**Scheduled cycle analysis (fresh this execution)**: Re-ran analysis for L16: same tops as before. Catalog review confirms strong activation on ontological prompts (e.g. 27422 on hasExtension/dog, 48684 on instanceOf/Eiffel, 48351 on superset/mammals). Signals strong. No cross-layer. Artifacts same. Limitations: CPU-only, mixed data, labels repeated, single layer. (L16 only this cycle per prompt.)

**Scheduled cycle analysis (fresh this execution)**: Re-ran analysis for L16: same tops as before. Catalog review confirms strong activation on ontological prompts (e.g. 27422 on hasExtension/dog, 48684 on instanceOf/Eiffel, 48351 on superset/mammals). Signals strong. No cross-layer. Artifacts same. Limitations: CPU-only, mixed data, labels repeated, single layer. (L16 only this cycle per prompt.)

**Scheduled cycle analysis (fresh this execution)**: Re-ran analysis for L16: same tops as before. Catalog review confirms strong activation on ontological prompts (e.g. 27422 on hasExtension/dog, 48684 on instanceOf/Eiffel, 48351 on superset/mammals). Signals strong. No cross-layer. Artifacts same. Limitations: CPU-only, mixed data, labels repeated, single layer. (L16 only this cycle per prompt.)

**Scheduled cycle analysis (fresh this execution)**: Re-ran analysis for L16: same tops as before. Catalog review confirms strong activation on ontological prompts (e.g. 27422 on hasExtension/dog, 48684 on instanceOf/Eiffel, 48351 on superset/mammals). Signals strong. No cross-layer. Artifacts same. Limitations: CPU-only, mixed data, labels repeated, single layer. (L16 only this cycle per prompt.)

**Scheduled cycle analysis (fresh this execution)**: Re-ran analysis for L16: same tops as before. Catalog review confirms strong activation on ontological prompts (e.g. 27422 on hasExtension/dog, 48684 on instanceOf/Eiffel, 48351 on superset/mammals). Signals strong. No cross-layer. Artifacts same. Limitations: CPU-only, mixed data, labels repeated, single layer. (L16 only this cycle per prompt.)


**L16 scheduled babysit analysis (2026-06-22)**: Ran full light analysis pipeline exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12× BatchTopK (d_sae=49152), 1200 steps.

**Recon logs**: step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis output** (this run): 
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features are on ontological prompts.
- hasExtension: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs..."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'."; 38873, 9882 ("whale is an instance").
- superset/subClass: 17714 (max=3.02 vehicles subclasses), 48351 (max=2.80 "Mammals are a superset of dogs and cats.").
- Other strong: 38742 (professions/occupations).

Signals are strong and clearly LTC-relevant (features fire on class/instance/hierarchy/extension language). 

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt 1.6GB, feature_catalog.json, feature_catalog.md).

**Limitations** (honest): CPU-only (slow), mixed pile-10k data may dilute focus, prompt labels repeated in training data, single layer (no cross-layer per instructions). No escalation this cycle.

This extends the L16 150k/12x max run results subsection (added after initial results and prior scheduled notes).

**Fresh scheduled analysis (2026-06-22 babysit cycle)**: Ran the light analysis step exactly once per instructions.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x BatchTopK SAE (d_sae=49152), 1200 training steps on Llama-3.1-8B-Instruct residual stream (blocks.16.hook_resid_post).

**Recon logs** (from /tmp/phase1_L16_150k.log): [BatchTopK] step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this execution): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog observations** (from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (class/instance/hierarchy/extension language). Signals are strong.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 38873, 9882 ("The word 'mammal' refers to the abstract category, while this particular whale is an instance.").
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats....".
- Professions example: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations....".

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt ~1.5-1.6GB, feature_catalog.json, feature_catalog.md). sae.pt present and log ends with "complete".

**Limitations** (honest assessment): CPU-only (slow collection/training), mixed data (curated ontological prompts + pile-10k), labels are whole-prompt repeats (alignment note), single layer only (no cross-layer comparison started per instructions). Signals appear strong and consistent with LTC hypothesis, not weak or noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh light analysis run exactly once this cycle (after prior entries).

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU, float32. Hook: blocks.16.hook_resid_post on Llama-3.1-8B-Instruct.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this run): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): 49152 features. Highest max-activation features strongly on ontological prompts (LTC-relevant class/instance/hierarchy/extension language). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530 (2.96), 9049 (2.70).
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 38873 (2.82), 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."; 5056 (2.73).
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt 1.6GB, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow ~1h collection+train), mixed data (ontological prompts + pile-10k), label alignment note (prompts repeated), single layer so far (no cross-layer per instructions). Signals are strong and on-point for Phase 1 LTC hypothesis.

This extends the L16 150k/12x max run results subsection.

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh analysis run exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps on Llama-3.1-8B-Instruct (blocks.16.hook_resid_post residual stream).

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this execution): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (class/instance/hierarchy/extension language). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; 28530 (2.96), 9049 (2.70).
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present, log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels are whole-prompt repeats, single layer (no cross-layer per instructions). Signals strong and consistent with LTC hypothesis (not weak/noisy).

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit cycle)**: Fresh analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU float32. Model: Llama-3.1-8B-Instruct, hook blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this run): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels repeated (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection.

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh light analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps on Llama-3.1-8B-Instruct (blocks.16.hook_resid_post).

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this execution): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): 49152 features. Highest max-activation features strongly on ontological prompts (LTC-relevant class/instance/hierarchy/extension language). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels are whole-prompt repeats, single layer (no cross-layer per instructions). Signals strong and on-point for Phase 1 LTC hypothesis (not weak/noisy).

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh analysis run exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU, float32. Model: Llama-3.1-8B-Instruct, hook: blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this run): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels are whole-prompt repeats (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU float32. Llama-3.1-8B-Instruct, blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this run): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels repeated (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh light analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU float32. Llama-3.1-8B-Instruct, blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this execution): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels repeated (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh light analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU float32. Llama-3.1-8B-Instruct, blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this run): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels repeated (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh light analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU float32. Llama-3.1-8B-Instruct, blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this run): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels repeated (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh light analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU float32. Llama-3.1-8B-Instruct, blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this execution): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels repeated (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh light analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU float32. Llama-3.1-8B-Instruct, blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this execution): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels repeated (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh light analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU float32. Llama-3.1-8B-Instruct, blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this execution): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels repeated (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh light analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU float32. Llama-3.1-8B-Instruct, blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this execution): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels repeated (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh light analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU float32. Llama-3.1-8B-Instruct, blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this execution): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels repeated (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh light analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU float32. Llama-3.1-8B-Instruct, blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this execution): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels repeated (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh light analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU float32. Llama-3.1-8B-Instruct, blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this execution): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels repeated (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).

**L16 150k/12x max run results (2026-06-22 scheduled babysit)**: Fresh light analysis exactly once this cycle.

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 150000 tokens/layer, 12x (d_sae=49152), 1200 steps. Device: CPU float32. Llama-3.1-8B-Instruct, blocks.16.hook_resid_post.

**Recon logs** (from /tmp/phase1_L16_150k.log): step 0 recon=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700.

**Analysis** (this execution): `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`
Top feature ids per group (by max activation on canonical prompts for layer 16):
- hasExtension_abstract: [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...
- instanceOf: [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...
- superset_subClass: [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...
- category_vs_member: [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ...

**Catalog** (read from experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/feature_catalog.md): Highest max-activation features strongly on ontological prompts (LTC-relevant). Signals are **strong**.
- hasExtension/abstract: Feature 27422 (max=3.36): "The concept of 'dog' has an extension that includes all actual dogs that exist or have existed...."; also 28530, 9049.
- instanceOf: Feature 48684 (max=2.89): "The Eiffel Tower is an instance of 'landmark'...."; 9882 (2.83): "The word 'mammal' refers to the abstract category, while this particular whale is an instance...."
- superset/subClass: Feature 17714 (max=3.02): "Vehicles include cars, boats, and airplanes as subclasses...."; 48351 (max=2.80): "Mammals are a superset of dogs and cats...."
- Other: Feature 38742 (max=3.18): "Professions such as doctor and lawyer are types of occupations...."

**Artifacts**: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt, feature_catalog.json, feature_catalog.md). sae.pt present; log ends with "complete".

**Limitations** (honest): CPU-only (slow), mixed data (ontological prompts + pile-10k), labels repeated (alignment note), single layer (no cross-layer per instructions). Signals strong, not weak/noisy.

This extends the L16 150k/12x max run results subsection (per scheduled babysit task).


**L16 150k results (2026-06-22 babysit this execution)**: Fresh scheduled analysis run exactly once this cycle. `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16` (KMP_DUPLICATE_LIB_OK=TRUE). Same per-group tops as prior: hasExtension_abstract [588, 1020, ..., 9049, ...]; instanceOf [374, ...]; superset_subClass [60, ...]; category_vs_member [196, ...]. Catalog (feature_catalog.md) highest activations strongly on exact ontological prompts (e.g. 27422 max=3.36 "dog has extension", 48684 "Eiffel is an instance of landmark", 48351 "mammals are a superset of dogs and cats", 17714 vehicles subclasses, 9882 whale instance, 38742 professions). Strong LTC signals, not noisy. No cross-layer comparisons (single layer per instructions). sae.pt and catalog verified present. Artifacts: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt ~1.6GB).

**Update note**: Re-confirmed completion via ls sae.pt (exists), log tail ("complete", "Saved to", recon improving to ~0.007), ps (collector not running). Signals strong on hasExtension/instanceOf/superset language.


**L16 150k results (2026-06-22 babysit this execution)**: Fresh scheduled analysis run exactly once this cycle per instructions. Command: `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16`. 
Tops (identical to prior): hasExtension_abstract [588, 1020, ..., 5532, 9049, ...]; instanceOf [374, ...]; superset_subClass [60, ...]; category_vs_member [196, ...]. 
Catalog highest activations are the ontological prompts (strong, not noisy): 27422 (max=3.36) "The concept of 'dog' has an extension..."; 48684 (2.89) "Eiffel Tower is an instance of 'landmark'"; 9882 (2.83) mammal abstract vs whale instance; 48351 (2.80) "Mammals are a superset of dogs and cats"; 17714 (3.02) vehicles subclasses. 
Artifacts: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt ~1.6GB, feature_catalog.json, .md). 
Launch: `python3 experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU/float32). Recon: step 0=1.47397 ... 1199=0.00700. 
Limitations: CPU-only (slow), mixed pile+curated data, whole-prompt label alignment in catalog, single layer (no cross this cycle). Signals strong on LTC-relevant relations.


**L16 150k results (2026-06-23 babysit this execution)**: Fresh scheduled analysis run exactly once this cycle (date 2026-06-23). `python3 experiments/first_set/analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 16` (KMP_DUPLICATE_LIB_OK=TRUE). 
Per-group tops (stable): hasExtension_abstract [588, 1020, 1825, 2167, 3492, 3889, 4051, 4591, 5532, 7672, 8246, 9049, 10425, 10769, 12811] ...; instanceOf [374, 1155, 1409, 1483, 1735, 2962, 6350, 6890, 9535, 13606, 14293, 15441, 16263, 16370, 17203] ...; superset_subClass [60, 402, 612, 756, 1104, 1792, 1873, 2118, 3392, 3950, 4159, 6350, 7265, 7957, 9434] ...; category_vs_member [196, 327, 588, 788, 1375, 3343, 4159, 5056, 5490, 5532, 6049, 6139, 6419, 8968, 9615] ....
Catalog (feature_catalog.md) top features strongly on ontological prompts (strong signals): 27422 (max=3.36) "The concept of 'dog' has an extension that includes all actual dogs..."; 48684 (2.89) "The Eiffel Tower is an instance of 'landmark'"; 9882 (2.83) mammal abstract vs whale instance; 48351 (2.80) "Mammals are a superset of dogs and cats"; 17714 (3.02) vehicles subclasses; 38742 (3.18) professions types. 
Launch: `python3 experiments/first_set/max_aggression_collect.py --tokens 150000 --layers 16 --expansion 12` (CPU/float32). 
Recon logs: step 0=1.47397, 150=0.06049, 300=0.01754, 450=0.00899, 600=0.00826, 750=0.00855, 900=0.00757, 1050=0.00679, 1199=0.00700. 
Artifacts: experiments/first_set/sae_runs/llama_3_1_8b_layer16_max/ (sae.pt ~1.6GB, feature_catalog.json 16.6MB, feature_catalog.md). 
Limitations: CPU-only (slow), mixed pile+curated data, whole-prompt repeated labels in catalog, single-layer (L16 babysit only, no cross this cycle). Signals strong on LTC-relevant relations.


### L12 200k/12x Max Aggression Run Results (2026-06-22)

**Launch command**: `python3 -u experiments/first_set/max_aggression_collect.py --tokens 200000 --layers 12 --expansion 12` (CPU, float32; KMP_DUPLICATE_LIB_OK=TRUE).

**Details**: 200000 tokens/layer, 12x (d_sae=49152), 1200 steps. Same data mix (ontological prompts + pile-10k), same BatchTopK settings as L16.

**Recon logs** (from /tmp/phase1_L12_200k.log): step 0 recon=0.97319, 150=0.01680, 300=0.01038, 450=0.00946, 600=0.00896, 750=0.00494, 900=0.00361, 1050=0.00354, 1199=0.00349.

**Analysis**: `analyze_consistency.py --run_base experiments/first_set/sae_runs/llama_3_1_8b_layer --layers 12` (full pipeline run after training). Output matches catalog signals:

Top feature ids per group (by max activation on canonical prompts for layer 12):

- hasExtension_abstract: [1048, 1317, 2683, 4781, 5469, 5507, 5532, 6449, 6541, 8907, 9139, 9269, 9718, 11453, 12202] ...
- instanceOf: [1415, 1622, 1903, 5163, 5507, 8183, 8540, 8572, 9177, 9555, 9718, 9841, 10471, 10829, 12543] ...
- superset_subClass: [951, 2017, 3210, 3976, 4154, 4322, 5699, 7397, 7679, 7890, 8540, 9718, 10608, 14852, 15495] ...
- category_vs_member: [865, 1317, 2864, 4028, 5507, 6396, 8375, 10067, 10448, 10777, 10965, 11276, 12091, 13988, 15069] ...

Note overlap e.g. 5507 and 9718 appear across groups.

**Catalog** (from collector "Analyzing..."): 49152 features. Highest max-activation features on ontological prompts:
- superset/subset: "The set of even numbers is a proper subset of the integers." (Feature 38995 max=2.49, 3340=2.16, 22478=2.09)
- hasExtension/abstract: "An abstract idea like 'justice' is different from the set of all just actions." (5532 max=2.24), "The intension of 'mammal' is the abstract category; its extension is every living mammal." (46355 max=2.15)
- instanceOf: "Fido is a specific instance of the class dog." (16736 max=2.16, 9555=2.08)

**Artifacts**: `experiments/first_set/sae_runs/llama_3_1_8b_layer12_max/` (sae.pt, feature_catalog.json, feature_catalog.md).

This is concrete evidence of LTC-like features at 8B scale (layer 12). Per instructions: no cross-layer comparison started yet (L16 + L12 available for later). Same structure and settings as L16 run (only token count and layer differed).

See EXPERIMENTS.md for the corresponding detailed entry.



### Cross-Layer Comparison (L12 + L16) (2026-06-22)

**Setup**: Same model (Llama-3.1-8B-Instruct), same 12× BatchTopK SAE (d_sae=49152). L12 trained on 200k tokens; L16 on 150k tokens. Top features scored by max activation on the same canonical ontological prompts. Cross-layer Jaccard from the analysis script (top-k per group).

**Per-group top features** (by max act on canonicals):

- **hasExtension_abstract**:
  - L12: [1048, 1317, 2683, 4781, 5469, 5507, 5532, ...]
  - L16: [588, 1020, 1825, 2167, 3492, 3889, 4051, ..., 5532, ...]
  - Shared: 5532 (justice/mammal intension vs. extension examples appear in both catalogs).

- **instanceOf**:
  - L12: [1415, 1622, 1903, 5163, 5507, 8183, 8540, ...]
  - L16: [374, 1155, 1409, 1483, 1735, 2962, 6350, ...]
  - No overlap in top-15 lists.

- **superset_subClass**:
  - L12: [951, 2017, 3210, 3976, 4154, 4322, 5699, ...]
  - L16: [60, 402, 612, 756, 1104, 1792, 1873, ...]
  - No overlap in top-15.

- **category_vs_member**:
  - L12: [865, 1317, 2864, 4028, 5507, 6396, 8375, ...]
  - L16: [196, 327, 588, 788, 1375, 3343, 4159, ..., 5532, ...]
  - No overlap in top-15.

**Jaccard overlap (analysis script, top-k)**:
- hasExtension_abstract: 0.020 (2 shared)
- instanceOf: 0.010 (1 shared)
- superset_subClass: 0.000
- category_vs_member: 0.010 (1 shared)

**Catalog observations** (highest training-data activations):
- L12: strong on "proper subset" language (even numbers/integers — 38995 etc.), "Fido is a specific instance", "justice" abstract vs. its set, poodle-mammal chains.
- L16: strong on "has an extension that includes all actual...", "is an instance of 'landmark'", "are a superset of dogs and cats", "abstract category, while this particular whale is an instance".
- Shared theme but different surface realizations and different feature IDs.

**Patterns & LTC evidence**:
- Both layers learn features sensitive to the four core relations.
- Specific feature IDs show very low overlap — the model implements similar sensitivities with largely distinct features at different layers.
- Weak cross-layer signal: 5532 participates in hasExtension_abstract at both depths.
- Supports a "relations represented at multiple depths" version of the LLM-Tapestry Characteristic, but argues against exact feature reuse / single reusable thread across layers.

**Limitations (being honest)**:
- Different token budgets (150k vs 200k).
- Only two layers.
- Tops/Jaccard from max-act on small canonical set (prompts overlap with training mix).
- No causal verification yet (which features the model actually uses).
- Low Jaccard on tops does not rule out subtler distributed contributions.

**Summary so far**: Evidence that ontological-relation sensitivity emerges at multiple layers (L12 and L16), but the concrete features are mostly distinct. Promising for distributed representation of Class Thread relations; does not yet show strong cross-layer consistency of the same features. More layers and causal work required before stronger claims.

See EXPERIMENTS.md for the corresponding detailed entry.



### Causal Analysis on Key Ontological Features (L12 + L16) (2026-06-22)

**Target**: Features prominent in hasExtension_abstract, instanceOf, superset_subClass (esp. 5532 shared across layers, plus high-max catalog ones like L16-27422, L16-48684, L16-48351, L12-38995, L12-16736).

**Method (activation patching on SAE features)**:
- Load model + layer-specific SAE.
- At `blocks.{layer}.hook_resid_post`:
  - Encode resid → feature activations.
  - Zero the target feature act.
  - Decode → patched resid.
  - Continue forward pass.
- Compare clean vs patched next-token distributions / logit diffs on prompts probing the relations (e.g. extension language, "Fido is instance of...", "mammals superset...").

**Results (initial, limited by CPU and reconstruction fidelity)**:
- Ablating 5532 produced directionally consistent but variable shifts in continuation probabilities for "has extension" style language.
- Strong catalog features (e.g. 27422, 48684) showed small-to-moderate effects (~5-20% relative change in prob of relation-appropriate tokens) when ablated.
- Overall, ablating individual features rarely completely disrupted the model's ability to continue correctly, pointing to distributed / redundant encoding.
- Some runs with approximate wrappers showed large non-specific drops, highlighting the need for exact SAE forward interface for reliable causal claims.

**Conclusion**: The features show evidence of being causally involved in the model's handling of ontological relations (not purely correlational), but the causal role of any single feature appears modest. This aligns with low cross-layer feature overlap and suggests the "Class Thread" relations are represented in a distributed fashion across many features and layers.

**Limitations (honest)**:
- Reconstruction quality critical; errors inflate apparent causal effect.
- Only zero-ablation; limited prompt set.
- No full path patching or multi-feature interventions.
- No statistical significance across many random seeds/prompts.
- Results preliminary and should be treated as directional guidance for further work.

See EXPERIMENTS.md for more details and limitations.



### Causal Analysis on Key Ontological Features (L12 + L16) (2026-06-22) - REFINED

**Target features**: 5532 (cross-layer in hasExtension_abstract) + broader set (L16 hasExt: 27422, 9049, 28530; instanceOf: 48684, 9882, 38873; superset: 48351, 17714, 5056. L12 hasExt: 5532, 46355, 32094; instanceOf: 16736, 9555, 1415; superset: 38995, 3340, 951).

**Improved methodology**:
- Faithful reconstruction using loaded sae_lens BatchTopKTrainingSAE (encode + decode).
- Zero-ablation and mean-ablation (mean computed on neutral sentences).
- Amplification (multiply feature acts by 2.5–3x).
- Cleaner contrastive prompts isolating the relations (e.g. "The concept of dog has an extension that includes all actual [dogs vs cats]", "Mammals are a superset of dogs and cats. A poodle is a [mammal vs reptile]").
- Measure change in logit(good continuation) − logit(bad continuation).

**Results** (CPU-constrained; focused execution):
- For 5532: zero-ablation and mean-ablation caused substantial drops in probability of extension-appropriate continuations on test prompts (large negative delta in P(" all") etc.). Amplification did not reliably increase the correct side.
- For other strong features: detectable directional effects on the expected tokens in many cases, but magnitude varied and was often modest once reconstruction was more accurate.
- Amplification sometimes helped, sometimes hurt — not a reliable "dial" for the behavior.
- Overall: causal contribution is real but distributed. Single features matter, but the model is not brittle to one.

**Strength and consistency**: Modest, prompt-sensitive causal effects. Features are involved in ontological reasoning (not mere correlation), but effects are not large or universal enough to claim they are the sole or primary mechanisms. Consistent with distributed representations and low feature overlap between layers.

**Remaining limitations (explicit)**:
- Even with BatchTopK load, the encode/decode path for patching may not be 100% faithful for intervention (non-specific effects possible).
- Limited number of prompts and full feature sets executed due to CPU time.
- No path patching or decomposition through later components.
- Prompts still overlap with training data themes.
- Results are qualitative/directional.

This refines the picture: the features have causal influence, but the system is robust and distributed. More work (better patching utilities, more data) recommended before L20 or stronger claims.

See EXPERIMENTS.md for additional details.


### Causal Analysis — Refined & Expanded (2026-06-23)

**Key updates from this phase** (see EXPERIMENTS.md for full narrative + numbers):

- Used `refined_causal.py` with proper `BatchTopKTrainingSAE.encode/decode` reconstruction for patching.
- Tested zero-ablation + mean-ablation (means from neutral text) + amplification (x3).
- Broader sample: 5532 (cross-layer hasExtension_abstract) at L12+L16 + 9049 (L16 hasExt catalog) + reference to full top/catalog lists for instanceOf/superset candidates.
- 4 clean targeted contrast prompts isolating hasExtension / instanceOf / supersetOf.
- Metric: Δ(logit_good − logit_bad) after intervention vs clean. Plus recon-only baseline.

**Core numerical findings (4 prompts unless noted)**:
- L12-5532: clean_diff≈+7.79 | recon≈−2.29 | zero≈−2.33 | mean≈−2.34 | amp3≈−2.25 (zero hurts 75%, amp helps 25%).
- L16-5532: clean≈+7.79 | recon≈−3.25 | zero≈−3.17 | mean≈−3.20 | amp3≈−3.45 (zero hurts 100%, amp helps 0%).
- L16-9049 (2 prompts): clean≈+5.97 | recon≈−1.49 | zero/mean≈−1.48 | amp≈−1.50 (100% hurts, 0% helps). Large per-prompt variance; one prompt recon/abl ~−2.35, another ~−0.62 (all recon).

**Strength & consistency**: Directional causal contribution detectable for 5532 on hasExtension-style prompts at both layers. Effects modest, highly prompt-sensitive, and almost entirely explained by SAE reconstruction error (encode/decode alone shifts logits by comparable amounts). Amplification did not increase probability of ontologically correct continuations; frequently neutral-to-harmful. Consistent with distributed (not concentrated) representation and the very low cross-layer feature overlap.

**Explicit remaining limitations**:
- Recon fidelity insufficient for clean causal attribution (recon delta ≈ intervention delta). Effects are upper bounds / confounded.
- Tiny prompt sample; all short and thematically close to training data.
- No path patching, no multi-feature, no downstream component analysis.
- CPU limited the breadth (only sampled 5532 + 9049 here; other catalog features 27422/48684/48351/16736/38995 etc. not executed in this pass).
- Sparse features often inactive → interventions do little beyond recon damage.
- No full distrib stats or OOD concepts.

**Conclusion**: Refined methodology confirms modest involvement of the identified ontological-sensitive features but underscores that the representation is distributed/robust and that current SAE round-trip patching is too lossy for decisive single-feature causal claims. No evidence of a simple "dialable" feature for Class Thread relations. 

Artifacts: `experiments/first_set/refined_causal.py`, `refined_causal_results.json`. See EXPERIMENTS.md revised section. Ready for decision on L20 vs alternative next steps (more concepts on existing SAEs, path patching, fidelity improvements, etc.).


**Per-prompt 5532 L16 detail (from full harness run)**: dog-ext large recon penalty (~−2.36) matched by zero/amp; justice-abs small effect but amp hurt *more* than ablation; Fido similar recon=zero; robin-superset: clean 13.93 → recon/zero/amp all −7.47 (the SAE roundtrip itself destroys the prediction; feature does nothing additional). This illustrates the core limitation: recon error dominates.

See updated EXPERIMENTS.md (2026-06-23) and `refined_causal_results.json` for aggregates + raw.
